# Loka Research Agent - Strands Version

Build a research agent about **Loka** step by step with
**[Strands Agents](https://strandsagents.com/)**.

This notebook is a **guided hands-on exercise** that you will follow throughout the workshop. The agent will be built progressively in three exercises, each adding more capabilities. You will find the following markers in the notebook to guide you:

| Marker               | Meaning                                              |
|----------------------|------------------------------------------------------|
| 🛠️ **Setup**        | Run this once to set up the environment and imports. |
| ✅ **Given**          | Code or instructions provided for you                |
| ✏️ **Your turn**     | Code you need to write to complete the exercise.     |
| 🚀 **Going further** | Optional stretch ideas if you finish early.          |
| 📚 **Hints**         | Links into the Strands docs for more information.    |

The exercises contain detailed instructions on what to do, but you are encouraged to explore and experiment. The goal is to learn by doing, so feel free to modify the code and see how it behaves. Refer to the Strands documentation for more information on the concepts and APIs used in this notebook. Make sure to switch on the `Python` toggle in the top right of documentation web page to see examples that match the code in this notebook.

Sample solutions will be provided in the `solutions/` folder after each exercise's experiment time. Use `git pull` to get them.

## 🛠️ Setup

Run this cell once to set up the environment and imports. Make sure you have already configured an `.env` file with your **Anthropic API key** and run `uv sync` to have the project dependencies installed. If you haven't done this yet, follow the instructions in the `README.md` file.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root
ROOT = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / "shared").is_dir())
sys.path.insert(0, str(ROOT / "shared"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Add ANTHROPIC_API_KEY to your .env (copy .env.example)."

from strands import Agent, tool
from strands.models.anthropic import AnthropicModel
from knowledge_base import search_documents, list_topics
from website import search_website

model = AnthropicModel(
    client_args={"api_key": os.environ["ANTHROPIC_API_KEY"]},
    model_id=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
    max_tokens=1024,
    params={"temperature": 0.3},
)

print("Model ready:", model.config["model_id"])

## ✅ The knowledge base (given)

Your agent's knowledge lives in `shared/`, already written for you:
`search_documents(query)`, `list_topics()`, and `search_website(query)`. In the
exercises you'll wrap these as **tools**. Run this to see what they return (no API
key needed):

In [ ]:
print(list_topics())
print("\n--- search_documents('learning') ---\n")
print(search_documents("learning"))

In [ ]:
print("--- search_website('services') ---\n")
print(search_website("what customers does Loka work with?"))

## Exercise 1: Basic Agent

An agent is a **model** + **tools** + **instructions**. You hand it a tool and *the model decides* when to call it.

Your job: turn the given `search_documents` function into a tool, build the agent, and run it. The system prompt is written for you.

> 💡 The tool's **docstring** is what the model reads to decide when to use it — write it for the model.

**📚 Hints**
- [Defining tools with `@tool`](https://strandsagents.com/docs/user-guide/concepts/tools/#building--loading-tools)
- [Agent API reference](https://strandsagents.com/docs/api/python/strands.agent.agent/)

### ✏️ Your turn

Build a **basic agent** that can answer questions about Loka's internal knowledge base. The model should decide when to call the tool you create.

In [ ]:
from strands import Agent, tool

SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

- Always answer from the knowledge base via your search tool. Don't rely on prior knowledge about Loka.
- If the knowledge base has no answer, say so instead of guessing.
- Be concise, warm, and a little proud of how great Loka is to work at."""


# TODO 1 — Make a tool.
#   Wrap the given `search_documents(query)` as a Strands @tool and add a docstring for the model to read.


# TODO 2 — Build the agent.
#   Fill in the Agent with `model`, your new tool, and SYSTEM_PROMPT.
agent = Agent()


# TODO 3 — Run it.
#   Ask "What is Loka's time-off policy?" and print the response.

### 🎯 Ask your own

In [ ]:
# TODO - Use the created agent to ask your own question about Loka.

### 🚀 Going further

If you finished early or want to explore more, try these ideas:

- **Inspect the run.** The call returns an `AgentResult`. Look at `result.message`,
  `result.metrics` (token usage), and `result.stop_reason`.
- **Use a [built-in Strands tool](https://strandsagents.com/docs/user-guide/concepts/tools/#2-vended-tools)** instead of a custom one. For example, add `strands_tools.calculator` to your agent's `tools=[...]` and ask it something math-related. See the
  [community tools package](https://strandsagents.com/docs/user-guide/concepts/tools/community-tools-package/) for more detail.
- **Change the persona** in `SYSTEM_PROMPT` (formal? pirate?) and re-run.
- **Ask something NOT in the knowledge base** and see if the agent admits it doesn't know.

## Exercise 2: Multi-turn Agent

Two complex patterns that upgrade the basic agent built in Exercise 1:
1. **Memory** — The agent remembers the conversation and can answer follow-ups. Strands keeps the messages for you, so you don't have to manage the state yourself. The model decides when to use it.
2. **Tool selection** — The agent can choose between multiple tools. You add a second tool, and the model decides which to call.

> 💡 Use `agent.messages.clear()` to reset the memory if you want to start a new conversation.

### ✏️ Your turn

Add two new tools to the agent you built in Exercise 1 and rebuild it so it can answer questions about both Loka's internal knowledge base and its public website. The model should decide which tool to call for each question and remember the conversation for follow-up questions.

In [ ]:
# Updated instructions
SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

Pick the right tool:
- the knowledge base tool: internal facts about working at Loka (benefits, time \
off, remote culture, awards, learning).
- the website tool: public info (services and solutions Loka offers, job \
openings, how it positions itself).

It's a conversation: use earlier turns to resolve follow-ups ("that", "there").
If neither source has the answer, say so. Be concise, warm, and proud of Loka."""


# TODO 1 — Add a second tool.
#   Wrap the given `list_topics()` as a Strands @tool and add a docstring for the model to read.


# TODO 2 — Add a third tool.
#   Wrap the given `search_website(query)` as a Strands @tool and add a docstring for the model to read.


# TODO 3 — Rebuild `agent` with BOTH tools.
#   Include your tool from Exercise 1, the two new tools, and the updated SYSTEM_PROMPT.
#   Is it necessary to indicate memory handling explicitly?

### ✅ Run a multi-turn conversation

Run the following three questions in order. Each prints just the answer. Notice how different questions use different tools, and that the third question works only because the agent remembers the second.

In [ ]:
response = agent("What can you tell me about Loka?")
print(response)

In [ ]:
print("tools used so far:", list(response.metrics.tool_metrics))

In [ ]:
response = agent("What has AWS recognized Loka for?")

In [ ]:
print("tools used so far:", list(response.metrics.tool_metrics))  # expect: the knowledge base

In [ ]:
response = agent("How would that show up in the projects engineers work on?") # requires memory

In [ ]:
print("tools used so far:", list(response.metrics.tool_metrics))

In [ ]:
print(f"Memory holds {len(agent.messages)} messages — Strands kept them for us.\n")

### 🎯 Ask your own

In [ ]:
# TODO - Continue the conversation with your own questions about Loka. Try to get the agent to use both tools and remember the context.
#   If you start to see the model get confused, question if the memory needs to be reset or managed differently

### 🚀 Going further (optional)

If you finished early or want to explore more, try these ideas:

- **Try an ambiguous follow-up** and see which tool it picks.
- **Peek at memory.** Print `agent.messages` to see exactly how Strands stores the conversation.
- **Cap the history** with a
  [conversation manager](https://strandsagents.com/docs/user-guide/concepts/agents/conversation-management/)
  (e.g. a sliding window) and see what changes.
- **Modify the system prompt** to change the way the agent handles follow-up questions.
- **Play with the model parameters** (temperature, max_tokens) and see how it affects the agent's behavior.

## Exercise 3: Open-ended Reasoning

Strands has a model-driven approach to agent building. Because of this, you can build an agent that can do **open-ended research** without writing any loops or planners. The model itself decides how to reason, which tools to call, and when to stop.

**📚 Hints**
- [Agent API reference (the agent loop)](https://strandsagents.com/docs/api/python/strands.agent.agent/)
- [Tools overview](https://strandsagents.com/docs/user-guide/concepts/tools/)

### ✏️ Your turn

In [ ]:
# Instructions for open-ended research (given):
SYSTEM_PROMPT = """You are the Loka Research Agent, a thorough assistant that \
answers questions about Loka (the company).

For open-ended questions:
- Break the question into the separate things you need to find out.
- Gather evidence with the tools, drawing on BOTH sources (the internal \
knowledge base AND the public website). Search multiple times; search again if \
you're missing something.
- Only once you have enough evidence, synthesize one well-structured answer.

Base every claim on the evidence you gathered; if something isn't covered, say \
so. Be warm, concrete, and proud of Loka."""


# TODO 1 — Build `agent` with all of the tools built previously and the new SYSTEM_PROMPT.


# TODO 2 — Ask a BROAD, multi-part question that needs several searches across
#   both sources, then print the response. (Idea: weigh a Loka job offer across
#   the kind of work, technical growth, and work-life balance)
result = agent("query")
print(result)

### ✅ How much work did that take?

In [ ]:
print("Reasoning cycles:", result.metrics.cycle_count)
for name, t in result.metrics.tool_metrics.items():
    print(f"  {name}: {t.call_count} call(s)")
print("Tokens:", result.metrics.accumulated_usage)

### 🎯 Ask your own

In [ ]:
# TODO - Ask your own open-ended question about Loka and print the response. Try to get the agent to use both tools and remember the context.

### 🚀 Going further (optional)

- **Compare phrasings.** Ask the same thing three ways and compare the show_tool_calls` counts — does a more specific question mean fewer searches?
- **New tool.** Add a new custom tool with a capability you would like to see the model use, and see if it can reason about when to call it.
- **Try to interrupt the model.** Is the Strands agent able to ask for human input natively to continue the research?
- **Tune the model.** Change `max_tokens` or `temperature` where `model` is defined and observe the difference.
- **Find the gaps.** Ask something the sources only partly cover and see how honestly it flags what it doesn't know.

## Conclusion

You built a research agent that can answer questions about Loka, starting with a single lookup and evolving to open-ended research across multiple sources. The model itself handled the reasoning, tool selection, and memory management, allowing you to focus on defining the tools and instructions.

The code you wrote was minimal, yet the agent's capabilities grew significantly with each exercise. Strands abstracts away the complexity of managing state and control flow, letting the model do the heavy lifting. Prompt engineering and tool design were the main levers you used to shape the agent's behavior.

#### Questions for further reflection:
- How did the model's behavior change as you added more tools and memory? Did the latency and number of tool calls increase, and if so, was it worth it for the richer answers?
- How does this agent compare to a traditional programmatic approach where you would explicitly code the reasoning and control flow? What are the trade-offs in terms of flexibility, maintainability, and performance?
- When would you *want* to take that control back and make the steps explicit?